In [1]:
using LinearAlgebra
using Printf


In [36]:
abstract type FiniteGroup end
abstract type FiniteGroupElement end 
abstract type FintieGroupClass end
abstract type FiniteGroupRepresentation end

tetrahedral_symbol = [:E, :C2x, :C2y, :C2z, :C3a, :C3a2, :C3b, :C3b2, :C3c, :C3c2, :C3d, :C3d2]

struct TetrahedralElement <: FiniteGroupElement   
    sym::Symbol
    rep::Matrix{Int64}
    inv::Union{Nothing, Symbol}
    group
    function char2tet(c::Char)
        if c== 'a'
            return [1 0 0 0]
        elseif c== 'b'
            return [0 1 0 0]
        elseif c== 'c' 
            return [0 0 1 0]
        elseif c== 'd'
            return [0 0 0 1]
        else
            throw(ArgumentError("Invalid character: $c"))
        end
    end
    function TetrahedralElement(s::Symbol, nn::String, inv::Union{Nothing, Symbol} = nothing)
        @assert length(nn) == 4
        @assert all(c -> c in "abcd", nn)
        mat = zeros(Int64, 4, 4) 
        for i in 1:4
            mat[:, i] = char2tet(nn[i])
        end
        return new(s, mat, inv, TetrahedralGroup)
    end
end

Base.show(io::IO, t::TetrahedralElement) = print(io, "$(t.sym) [TetrahedralElement] : ", t.rep)


struct TetrahedralGroup<:FiniteGroup
    elements::Dict{Symbol, TetrahedralElement}
    function TetrahedralGroup()
        els = Dict(
        :E => TetrahedralElement(:E, "abcd", :E),
        :C2x => TetrahedralElement(:C2x, "badc", :C2x),
        :C2y => TetrahedralElement(:C2y, "dcba", :C2y), 
        :C2z => TetrahedralElement(:C2z, "cdab", :C2z), 
        :C3a => TetrahedralElement(:C3a, "adbc", :C3a2),
        :C3a2 => TetrahedralElement(:C3a2, "acdb", :C3a),
        :C3b => TetrahedralElement(:C3b, "cbda", :C3b2),
        :C3b2 => TetrahedralElement(:C3b2, "dbac", :C3b),
        :C3c => TetrahedralElement(:C3c, "dacb", :C3c2),
        :C3c2 => TetrahedralElement(:C3c2, "bdca", :C3c),
        :C3d => TetrahedralElement(:C3d, "bcad", :C3d2),
        :C3d2 => TetrahedralElement(:C3d2, "cabd", :C3d)
        )
        return new(els)
    end
end


function (p::TetrahedralGroup)(s::Symbol)
    @assert s in [:E, :C2x, :C2y, :C2z, :C3a, :C3a2, :C3b, :C3b2, :C3c, :C3c2, :C3d, :C3d2]
    return p.elements[s]
end

function inv(t::FiniteGroupElement) 
    if t.inv == nothing
        return nothing
    else
        x=t.group()
        return x(t.inv)
    end
end

function find_in_group_by_representation(G::T, rep) where T<:FiniteGroup
    for (key, value) in G.elements
        if value.rep == rep
            return G(key)
        end
    end
    return nothing
end


Base.:*(a::T, b::T) where T<:FiniteGroupElement = find_in_group_by_representation(a.group(), a.rep * b.rep)

function find_class(G::T) where T<:FiniteGroup
    elementset0 = [t.sym for (k, t) in Tet.elements]
    elementset = Set(elementset0)
    elementclass = []
    while length(elementset) > 0
        el = pop!(elementset)
        class = Set([el,])
        for g in elementset0
            push!(class, ((G(g)*G(el))*inv(G(g))).sym)
        end
        push!(elementclass, class)
        elementset = setdiff(elementset, class)
    end
    return elementclass
end
    

find_class (generic function with 1 method)

In [9]:
E=TetrahedralElement(:E, "abcd")

E [TetrahedralElement] : [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [10]:
E.group()

TetrahedralGroup(Dict{Symbol, TetrahedralElement}(:C3c => C3c [TetrahedralElement] : [0 1 0 0; 0 0 0 1; 0 0 1 0; 1 0 0 0], :C3b => C3b [TetrahedralElement] : [0 0 0 1; 0 1 0 0; 1 0 0 0; 0 0 1 0], :C3d => C3d [TetrahedralElement] : [0 0 1 0; 1 0 0 0; 0 1 0 0; 0 0 0 1], :C3a2 => C3a2 [TetrahedralElement] : [1 0 0 0; 0 0 0 1; 0 1 0 0; 0 0 1 0], :C2x => C2x [TetrahedralElement] : [0 1 0 0; 1 0 0 0; 0 0 0 1; 0 0 1 0], :C3d2 => C3d2 [TetrahedralElement] : [0 1 0 0; 0 0 1 0; 1 0 0 0; 0 0 0 1], :C3b2 => C3b2 [TetrahedralElement] : [0 0 1 0; 0 1 0 0; 0 0 0 1; 1 0 0 0], :C3c2 => C3c2 [TetrahedralElement] : [0 0 0 1; 1 0 0 0; 0 0 1 0; 0 1 0 0], :C3a => C3a [TetrahedralElement] : [1 0 0 0; 0 0 1 0; 0 0 0 1; 0 1 0 0], :E => E [TetrahedralElement] : [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]…))

In [17]:
Tet=TetrahedralGroup()

TetrahedralGroup(Dict{Symbol, TetrahedralElement}(:C3c => C3c [TetrahedralElement] : [0 1 0 0; 0 0 0 1; 0 0 1 0; 1 0 0 0], :C3b => C3b [TetrahedralElement] : [0 0 0 1; 0 1 0 0; 1 0 0 0; 0 0 1 0], :C3d => C3d [TetrahedralElement] : [0 0 1 0; 1 0 0 0; 0 1 0 0; 0 0 0 1], :C3a2 => C3a2 [TetrahedralElement] : [1 0 0 0; 0 0 0 1; 0 1 0 0; 0 0 1 0], :C2x => C2x [TetrahedralElement] : [0 1 0 0; 1 0 0 0; 0 0 0 1; 0 0 1 0], :C3d2 => C3d2 [TetrahedralElement] : [0 1 0 0; 0 0 1 0; 1 0 0 0; 0 0 0 1], :C3b2 => C3b2 [TetrahedralElement] : [0 0 1 0; 0 1 0 0; 0 0 0 1; 1 0 0 0], :C3c2 => C3c2 [TetrahedralElement] : [0 0 0 1; 1 0 0 0; 0 0 1 0; 0 1 0 0], :C3a => C3a [TetrahedralElement] : [1 0 0 0; 0 0 1 0; 0 0 0 1; 0 1 0 0], :E => E [TetrahedralElement] : [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]…))

In [20]:
Tet(:C2x)*Tet(:C2y)

C2z [TetrahedralElement] : [0 0 1 0; 0 0 0 1; 1 0 0 0; 0 1 0 0]

In [26]:
inv(Tet(:C3b2))

C3b [TetrahedralElement] : [0 0 0 1; 0 1 0 0; 1 0 0 0; 0 0 1 0]

In [ ]:
[t.sym for (k, t) in Tet.elements]

In [37]:
find_class(Tet)

4-element Vector{Any}:
 Set([:C2x, :C2z, :C2y])
 Set([:C3b, :C3d, :C3a, :C3c])
 Set([:C3d2, :C3b2, :C3c2, :C3a2])
 Set([:E])

In [30]:
qs = [t.sym for (k, t) in Tet.elements]
Tet(qs[1])*Tet(qs[2])*inv(Tet(qs[1]))

C3a [TetrahedralElement] : [1 0 0 0; 0 0 1 0; 0 0 0 1; 0 1 0 0]

In [31]:
qs

12-element Vector{Symbol}:
 :C3c
 :C3b
 :C3d
 :C3a2
 :C2x
 :C3d2
 :C3b2
 :C3c2
 :C3a
 :E
 :C2y
 :C2z

In [ ]:
Tet(qs[2]).group

In [ ]:
for k1 in tetrahedral_symbol
    for (i, k2) in enumerate(tetrahedral_symbol)
        result = Tet(k1).rep * Tet(k2).rep
        found_element = find_in_group_by_representation(Tet, result)
        if found_element !== nothing
            q = String(found_element.sym)
            if i == 1
                @printf("%12s", "\$"*q*"\$ &")
            end
            
            if length(q) == 1
                @printf("%12s", "\$"*q*"\$ &")
            else
                @printf("%12s", "\$"*q[1]*"_{"*q[2:end]*"}\$ &")
            end
            
        else 
            @printf("%6s", "-------") 
        end
    end
    print("\\\\ \n")
end